In [1]:
"""
generate_david_jones_data.py
================================================================
Synthetic data generator for the David Jones marketing database
(BUSA8090 portfolio project).

Produces `Data.sql` — multi-row INSERT statements that are referentially
consistent with Entities.sql and satisfy every CHECK / ENUM / UNIQUE / FK
constraint. The data is deliberately shaped so the 14 analytical queries
surface a coherent EOFY-improvement narrative for June 2026 planning:

    * WeChat    - fewer but high-value conversions  -> best ROAS / ROI, premium AOV
    * Instagram - cheap reach, highest CTR          -> lowest CPA, Phone-led
    * Facebook  - broad, solid mid-funnel           -> steady converter, Desktop-friendly
    * X         - weak CTR & conversion, dear CPA    -> over-funded in EOFY 2025 (cut it)
    * EOFY sales grow YoY, but 2025 ROI was dragged by the X over-allocation

Date span : 2021-01-01 .. 2026-05-31  (history up to the eve of EOFY 2026)
Reproducible: fixed seed. Re-running yields the identical dataset.

USAGE
-----
    pip install numpy
    python generate_david_jones_data.py        # writes ./Data.sql

Then in MySQL:
    SOURCE Entities.sql;
    SOURCE Data.sql;

Tune the dataset scale with the CONFIG knobs below.
================================================================
"""
import random
from datetime import date, datetime, timedelta
import numpy as np

# ---------------------------------------------------------------- CONFIG
SEED        = 42
N_CUSTOMERS = 400
N_PRODUCTS  = 100
N_ORDERS    = 10000
N_INTER     = 15000         # interaction_events
OUTPUT_PATH = "Data.sql"

random.seed(SEED)
np.random.seed(SEED)

START = date(2021, 1, 1)
END   = date(2026, 5, 31)

def d(x):  return x.strftime('%Y-%m-%d')
def dt(x): return x.strftime('%Y-%m-%d %H:%M:%S')
def rdate(a, b):
    return a + timedelta(days=random.randint(0, (b - a).days))

# ============================================================ 1. CUSTOMERS
LOCATIONS = {  # weight roughly by metro size / DJ store catchment
    'Sydney CBD NSW': 11, 'Bondi Junction NSW': 6, 'Chatswood NSW': 6,
    'Parramatta NSW': 5, 'Miranda NSW': 3, 'Hornsby NSW': 3, 'Macquarie NSW': 3,
    'Newcastle NSW': 3, 'Wollongong NSW': 2,
    'Melbourne CBD VIC': 10, 'Chadstone VIC': 6, 'Doncaster VIC': 4,
    'Highpoint VIC': 3, 'Geelong VIC': 2,
    'Brisbane CBD QLD': 7, 'Carindale QLD': 3, 'Indooroopilly QLD': 3,
    'Gold Coast QLD': 3, 'Adelaide CBD SA': 5, 'Perth CBD WA': 5,
    'Canberra ACT': 4, 'Hobart TAS': 2,
}
loc_names = list(LOCATIONS); loc_w = np.array(list(LOCATIONS.values()), float)
loc_w /= loc_w.sum()

FIRST = ['Olivia','Charlotte','Amelia','Mia','Isla','Grace','Chloe','Ava','Sophie','Ruby',
         'Jack','Oliver','William','Noah','Henry','Thomas','James','Lucas','Liam','Ethan',
         'Wei','Min','Hiroshi','Aarav','Priya','Mohammed','Fatima','Sofia','Diego','Elena',
         'Daniel','Emily','Hannah','Lily','Zoe','Ella','Leo','Max','Harper','Scarlett',
         'Johnny','Linh','Anh','Mai','Hung','Trang','Khoa','Ngoc','Bao','Quang']
LAST  = ['Smith','Jones','Williams','Brown','Wilson','Taylor','Nguyen','Tran','Le','Pham',
         'Chen','Wang','Li','Singh','Patel','Kumar','Khan','Garcia','Martin','Lee',
         'White','Walker','Harris','Clark','Lewis','Young','King','Wright','Scott','Green',
         'Anderson','Thompson','Roberts','Hall','Allen','Adams','Baker','Nelson','Carter','Mitchell']

customers = []
used_email, used_phone = set(), set()
for cid in range(1, N_CUSTOMERS + 1):
    fn, ln = random.choice(FIRST), random.choice(LAST)
    name = f"{fn} {ln}"
    # unique email
    while True:
        e = f"{fn.lower()}.{ln.lower()}{random.randint(1,9999)}@email.com"
        if e not in used_email: used_email.add(e); break
    # unique phone
    while True:
        p = "+614" + "".join(str(random.randint(0,9)) for _ in range(8))
        if p not in used_phone: used_phone.add(p); break
    # signup weighted toward earlier years (gives tenure for Q13/Q14)
    yr = np.random.choice([2021,2022,2023,2024,2025,2026], p=[.28,.22,.18,.15,.12,.05])
    lo = max(START, date(yr,1,1)); hi = min(END, date(yr,12,31))
    signup = rdate(lo, hi)
    tier = np.random.choice(['Bronze','Silver','Gold'], p=[.60,.28,.12])
    # ~30% referred by an earlier-id customer
    ref = random.randint(1, cid-1) if (cid > 1 and random.random() < 0.30) else None
    loc = np.random.choice(loc_names, p=loc_w)
    customers.append(dict(id=cid, name=name, email=e, loc=loc, phone=p,
                          ref=ref, signup=signup, tier=tier))

cust_by_id = {c['id']: c for c in customers}

# ============================================================ 2. PRODUCTS
# (category -> list of (brand, price_lo, price_hi, base_popularity))
CAT = {
    'Beauty':       [('Estee Lauder',60,260),('Clinique',45,160),('Chanel Beauty',80,320),
                     ('Dior Beauty',70,300),('Jo Malone',90,280),('La Mer',180,650)],
    'Womenswear':   [('Country Road',80,360),('Witchery',90,300),('Scanlan Theodore',200,650),
                     ('Aje',180,520),('Camilla',250,700),('Sass and Bide',140,480)],
    'Menswear':     [('Politix',120,420),('RM Williams',150,650),('Hugo Boss',180,750),
                     ('Ralph Lauren',120,520),('Tommy Hilfiger',90,360)],
    'Footwear':     [('Tony Bianco',140,320),('Mollini',130,260),('Nike',120,260),
                     ('Adidas',110,240),('Sol Sana',150,300)],
    'Home':         [('Sheridan',60,420),('Royal Doulton',40,380),('Maxwell and Williams',25,260),
                     ('Le Creuset',120,820),('Global',90,540)],
    'Electronics':  [('Dyson',299,1499),('Sonos',249,1299),('Breville',99,899),('Nespresso',129,799)],
    'Accessories':  [('Coach',180,950),('Michael Kors',120,650),('Oroton',150,720),('Fossil',90,420)],
    'Kids':         [('Seed Heritage',25,140),('Country Road Kids',30,160),('Polo Kids',40,180)],
    'Food and Wine':[('DJ Food Hall',18,160),('Penfolds',45,420),('Maison Wine',25,260)],
}
# category popularity weights (Beauty & Womenswear lead revenue/volume)
CAT_POP = {'Beauty':18,'Womenswear':18,'Menswear':12,'Footwear':12,'Home':11,
           'Electronics':7,'Accessories':10,'Kids':6,'Food and Wine':6}

products = []
pid = 0
cats = list(CAT)
# distribute 100 products across categories by popularity
alloc = {c: max(4, round(N_PRODUCTS * CAT_POP[c] / sum(CAT_POP.values()))) for c in cats}
for cat, n in alloc.items():
    for _ in range(n):
        if pid >= N_PRODUCTS: break
        brand, lo, hi = random.choice(CAT[cat])
        price = round(random.uniform(lo, hi), 2)
        margin = random.uniform(0.35, 0.62)          # gross margin
        cost = round(price * (1 - margin), 2)
        cost = max(1.0, cost)
        pid += 1
        products.append(dict(id=pid, name=f"{brand} {cat} Item {pid}", cat=cat,
                             brand=brand, price=price, cost=cost,
                             stock=random.randint(0, 500), pop=CAT_POP[cat]))
N_PRODUCTS = len(products)
prod_by_id = {p['id']: p for p in products}
prod_ids = [p['id'] for p in products]
prod_w = np.array([p['pop'] for p in products], float); prod_w /= prod_w.sum()

# ============================================================ 3. CAMPAIGNS
# recurring annual calendar; (label, start(m,d), end(m,d), base_budget, is_eofy)
TEMPLATE = [
    ('Boxing Day & New Year Sale', (12,26),(1,15), 380000, False),  # spans year boundary
    ('Autumn Fashion Launch',      ( 3, 1),(3,28), 180000, False),
    ('Mothers Day Gifting',        ( 4,20),(5,11), 240000, False),
    ('EOFY Stocktake Sale',        ( 6, 1),(6,30), 520000, True),
    ('Fathers Day Gifting',        ( 8,20),(9, 7), 220000, False),
    ('Spring Racing & Click Frenzy',(10,20),(11,18),420000, False),
]
campaigns = []
camp_id = 0
for yr in range(2021, 2026):                       # full EOFY years 2021..2025
    growth = 1.0 + 0.08 * (yr - 2021)              # ~8% YoY budget growth
    for label, (sm,sd), (em,ed), base, is_eofy in TEMPLATE:
        s = date(yr, sm, sd)
        e = date(yr if em >= sm else yr+1, em, ed)
        if s > END: continue
        e = min(e, END)
        camp_id += 1
        budget = round(base * growth * random.uniform(0.95,1.08), 2)
        obj = ('Drive clearance revenue & loyalty sign-ups' if is_eofy
               else 'Brand awareness & seasonal conversion')
        campaigns.append(dict(id=camp_id, name=f"{label} {yr}", obj=obj,
                              budget=budget, start=s, end=e,
                              is_eofy=is_eofy, year=yr,
                              # EOFY YoY efficiency: rises then dips in 2025 (the problem to fix)
                              eff={2021:.90,2022:1.00,2023:1.10,2024:1.18,2025:1.00}.get(yr,1.0)))
# 2026 partial year (no EOFY yet — that's the upcoming campaign to improve)
for label,(sm,sd),(em,ed),base,is_eofy in TEMPLATE:
    if is_eofy: continue
    s = date(2026, sm, sd)
    if s > END: continue
    e = min(date(2026, em, ed) if em>=sm else date(2026,em,ed), END)
    camp_id += 1
    campaigns.append(dict(id=camp_id, name=f"{label} 2026", obj='Brand awareness & seasonal conversion',
                          budget=round(base*1.45*random.uniform(.95,1.08),2),
                          start=s, end=e, is_eofy=False, year=2026, eff=1.0))
camp_by_id = {c['id']: c for c in campaigns}

# ============================================================ 4. ADVERTISEMENTS
# platform behaviour drives the whole funnel story
PLAT = {
    #          ctr range,    conv range,  impr/$  value/conv  device mix (Phone,Tablet,Desktop)
    'Instagram':dict(ctr=(.045,.075), conv=(.12,.18), ipd=55, vpc=6.0,  dev=[.62,.13,.25]),
    'Facebook': dict(ctr=(.030,.050), conv=(.15,.22), ipd=50, vpc=7.0,  dev=[.45,.15,.40]),
    'WeChat':   dict(ctr=(.022,.040), conv=(.20,.30), ipd=45, vpc=14.0, dev=[.55,.10,.35]),  # premium, high value
    'X':        dict(ctr=(.014,.028), conv=(.06,.12), ipd=42, vpc=4.0,  dev=[.58,.14,.28]),  # underperformer
}
plat_names = list(PLAT)

advertisements = []
ad_id = 0
for c in campaigns:
    # EOFY campaigns over-allocate to X in 2025 (the inefficiency to expose)
    if c['is_eofy'] and c['year'] == 2025:
        plats = ['Instagram','Facebook','WeChat','X','X']
    elif c['is_eofy']:
        plats = ['Instagram','Facebook','WeChat','X']
    else:
        plats = random.sample(plat_names, k=random.randint(3,4))
    # 1-2 creatives per chosen platform -> more ads / more performance rows
    plats = [pl for pl in plats for _ in range(random.randint(1, 2))]
    n_ads = len(plats)
    # split budget across ads with some noise
    raw = np.random.uniform(0.8,1.2,n_ads); share = raw/raw.sum()
    for i, pl in enumerate(plats):
        ad_id += 1
        abudget = round(float(c['budget'] * share[i]), 2)
        abudget = max(500.0, abudget)
        # ad window inside campaign window
        span = (c['end'] - c['start']).days
        a_s = c['start'] + timedelta(days=random.randint(0, max(0, span//4)))
        a_e = c['end'] - timedelta(days=random.randint(0, max(0, span//5)))
        if a_e <= a_s: a_e = c['end']
        advertisements.append(dict(id=ad_id, camp=c['id'], plat=pl, budget=abudget,
                                   start=a_s, end=a_e, year=c['year'],
                                   eff=c['eff'], is_eofy=c['is_eofy']))
ad_by_id = {a['id']: a for a in advertisements}

# ============================================================ 5. AD PERFORMANCE
perf = []
perf_id = 0
for a in advertisements:
    p = PLAT[a['plat']]
    run = max(1, (a['end'] - a['start']).days)
    periods = max(2, min(12, run // 4))         # report ~every 4 days, cap 12
    total_impr = a['budget'] * p['ipd'] * a['eff'] * random.uniform(0.9,1.1)
    for w in range(periods):
        rep = a['start'] + timedelta(days=int(run * w / periods) + random.randint(0,2))
        if rep > a['end']: rep = a['end']
        devs = np.random.choice(['Phone','Tablet','Desktop'],
                                size=random.randint(2,3), replace=False)
        for dev in devs:
            perf_id += 1
            impr = int(max(50, total_impr / periods / len(devs) * random.uniform(0.7,1.3)))
            ctr = random.uniform(*p['ctr'])
            clicks = max(1, int(impr * ctr))
            if clicks >= impr: clicks = impr - 1
            cr = random.uniform(*p['conv'])
            conv = int(clicks * cr)
            if conv >= clicks: conv = clicks - 1
            conv = max(0, conv)
            revenue = round(conv * p['vpc'] * random.uniform(0.8,1.2), 2)
            perf.append(dict(id=perf_id, ad=a['id'], rep=rep, dev=dev,
                             impr=impr, clicks=clicks, conv=conv, rev=revenue))

# ============================================================ 6. ORDERS + LINES
# seasonal month weights (June EOFY & Nov/Dec peak), year growth
MONTH_W = {1:1.1,2:.8,3:.9,4:1.0,5:1.2,6:1.7,7:.9,8:.9,9:1.1,10:1.0,11:1.4,12:1.6}
YEAR_W  = {2021:.8,2022:.95,2023:1.05,2024:1.15,2025:1.25,2026:0.9}

def pick_order_date():
    # weight (year, month) jointly
    while True:
        yr = np.random.choice(list(YEAR_W), p=np.array(list(YEAR_W.values()))/sum(YEAR_W.values()))
        mo = np.random.choice(list(MONTH_W), p=np.array(list(MONTH_W.values()))/sum(MONTH_W.values()))
        try:
            day = random.randint(1, 28)
            od = date(yr, mo, day)
        except ValueError:
            continue
        if START <= od <= END:
            h = random.randint(8,22); mi = random.randint(0,59)
            return datetime(od.year, od.month, od.day, h, mi, 0)

# customers weighted so Gold order much more, Silver more than Bronze
tier_order_w = {'Bronze':1.0,'Silver':2.0,'Gold':4.0}
cust_pick_ids = [c['id'] for c in customers]
cust_pick_w = np.array([tier_order_w[c['tier']] for c in customers], float)
# leave a few customers with zero orders (keeps LEFT JOINs meaningful, e.g. Q4 note)
zero_order = set(random.sample(cust_pick_ids, 8))
for i, cid in enumerate(cust_pick_ids):
    if cid in zero_order: cust_pick_w[i] = 0.0
cust_pick_w /= cust_pick_w.sum()

orders = []
order_products = []
oid = 0
for _ in range(N_ORDERS):
    oid += 1
    cid = int(np.random.choice(cust_pick_ids, p=cust_pick_w))
    tier = cust_by_id[cid]['tier']
    odt = pick_order_date()
    # signup must precede order; if not, push order after signup
    su = cust_by_id[cid]['signup']
    if odt.date() < su:
        odt = datetime.combine(rdate(su, END), datetime.min.time()) + timedelta(hours=random.randint(8,21))
    status = np.random.choice(['Completed','Returned','Cancel'], p=[.88,.08,.04])
    # Gold lean to DJ card; others mixed
    if tier == 'Gold':
        pay = np.random.choice(['David Jones Credit Card','Debit/Credit Card','Gift Card','Others'], p=[.50,.34,.10,.06])
    elif tier == 'Silver':
        pay = np.random.choice(['David Jones Credit Card','Debit/Credit Card','Gift Card','Others'], p=[.28,.52,.12,.08])
    else:
        pay = np.random.choice(['David Jones Credit Card','Debit/Credit Card','Gift Card','Others'], p=[.12,.62,.16,.10])
    orders.append(dict(id=oid, cust=cid, dt=odt, pay=pay, status=status))
    # line items 1..4; Gold buys more & pricier
    n_lines = np.random.choice([1,2,3,4], p=[.45,.32,.16,.07])
    chosen = set()
    tier_price_mult = {'Bronze':0.9,'Silver':1.0,'Gold':1.15}[tier]
    for _ in range(n_lines):
        p_id = int(np.random.choice(prod_ids, p=prod_w))
        if p_id in chosen:  # PK(order_id,product_id) must be unique
            continue
        chosen.add(p_id)
        pr = prod_by_id[p_id]
        qty = int(np.random.choice([1,2,3], p=[.72,.21,.07]))
        gross = round(pr['price'] * random.uniform(0.96,1.06) * tier_price_mult, 2)
        gross = max(1.0, gross)
        order_products.append(dict(order=oid, prod=p_id, qty=qty, gross=gross))

orders_by_id = {o['id']: o for o in orders}
# index order_products by order
op_by_order = {}
for op in order_products:
    op_by_order.setdefault(op['order'], []).append(op)

# campaign windows for attribution / discounts
def campaigns_active(on_date):
    return [c for c in campaigns if c['start'] <= on_date <= c['end']]

# ============================================================ 7. ADVERTISEMENT_ORDER (attribution)
ad_by_camp = {}
for a in advertisements:
    ad_by_camp.setdefault(a['camp'], []).append(a)

advertisement_order = []
ao_seen = set()
for o in orders:
    if o['status'] == 'Cancel':
        continue
    active = campaigns_active(o['dt'].date())
    if not active or random.random() > 0.72:     # ~62% of in-window orders attributed
        continue
    c = random.choice(active)
    ads = ad_by_camp.get(c['id'], [])
    if not ads: continue
    a = random.choice(ads)
    key = (a['id'], o['id'])
    if key in ao_seen: continue
    ao_seen.add(key)
    clicked = o['dt'].date() - timedelta(days=random.randint(0,3))
    if clicked < a['start']: clicked = a['start']
    advertisement_order.append(dict(ad=a['id'], order=o['id'], clicked=clicked))

# ============================================================ 8. INTERACTION_EVENTS
interaction_events = []
iid = 0
for _ in range(N_INTER):
    a = random.choice(advertisements)
    iid += 1
    cid = int(np.random.choice(cust_pick_ids, p=cust_pick_w))
    su = cust_by_id[cid]['signup']
    # datetime within ad window and after signup
    lo = max(a['start'], su); hi = a['end']
    if hi < lo:
        hi = a['end']; lo = a['start']
    base = rdate(lo, hi) if hi >= lo else a['start']
    it = datetime.combine(base, datetime.min.time()) + timedelta(hours=random.randint(7,23), minutes=random.randint(0,59))
    # Phone dominates count; Desktop longer dwell
    dev = np.random.choice(['Phone','Tablet','Desktop'], p=PLAT[a['plat']]['dev'])
    dur_base = {'Phone':28,'Tablet':40,'Desktop':62}[dev]
    dur = round(max(0.0, np.random.gamma(2.0, dur_base/2.0)), 2)
    interaction_events.append(dict(id=iid, ad=a['id'], cust=cid, dur=dur, dt=it, dev=dev))

# ============================================================ 9. DISCOUNTS
discounts = []
disc_id = 0
PROMO = ['EOFY','SAVE','VIP','SPRING','MUM','DAD','BOXING','LOYAL','EXTRA','FLASH']
for c in campaigns:
    n_d = 2 if c['is_eofy'] else random.randint(1,2)
    for _ in range(n_d):
        disc_id += 1
        dtype = np.random.choice(['Percentage','Fixed Amount','Free Shipping'], p=[.62,.30,.08])
        scope = np.random.choice(['Order','Line'], p=[.55,.45])
        if dtype == 'Percentage':
            val = round(float(np.random.choice([10,15,20,25,30,40])), 2)
        elif dtype == 'Fixed Amount':
            val = round(float(np.random.choice([20,30,50,75,100])), 2)
        else:
            val = 0.0
        promo = f"{random.choice(PROMO)}{random.randint(10,99)}"
        min_spend = round(float(np.random.choice([0,50,100,150,200])),2)
        ds = c['start']; de = c['end']
        discounts.append(dict(id=disc_id, camp=c['id'],
                              name=f"{c['name'].split(' 20')[0]} Offer",
                              dtype=dtype, scope=scope, val=val, promo=promo,
                              min_spend=(None if min_spend==0 else min_spend),
                              start=ds, end=de))
# a few standalone (campaign_id NULL) evergreen discounts
for nm in ['Welcome Member Offer','Birthday Treat','Online Exclusive']:
    disc_id += 1
    discounts.append(dict(id=disc_id, camp=None, name=nm,
                          dtype='Percentage', scope='Order', val=10.0,
                          promo=f"EVER{random.randint(10,99)}",
                          min_spend=50.0, start=START, end=END))
disc_by_id = {x['id']: x for x in discounts}
disc_by_camp = {}
for x in discounts:
    if x['camp'] is not None:
        disc_by_camp.setdefault(x['camp'], []).append(x)

# ============================================================ 10. ORDER DISCOUNTS
order_discount = []
order_product_discount = []
od_seen, opd_seen = set(), set()

for o in orders:
    if o['status'] == 'Cancel':
        continue
    active = campaigns_active(o['dt'].date())
    cands = []
    for c in active:
        cands += disc_by_camp.get(c['id'], [])
    if not cands or random.random() > 0.45:    # ~45% of in-window orders use a discount
        continue
    disc = random.choice(cands)
    lines = op_by_order.get(o['id'], [])
    if not lines: continue
    subtotal = sum(l['qty']*l['gross'] for l in lines)
    if disc['min_spend'] and subtotal < disc['min_spend']:
        continue
    if disc['scope'] == 'Order':
        if disc['dtype'] == 'Percentage':
            amt = subtotal * disc['val']/100.0
        elif disc['dtype'] == 'Fixed Amount':
            amt = min(disc['val'], subtotal*0.5)
        else:  # Free Shipping
            amt = round(random.uniform(8,18),2)
        amt = round(min(amt, subtotal*0.9), 2)
        key = (o['id'], disc['id'])
        if amt >= 0 and key not in od_seen:
            od_seen.add(key)
            order_discount.append(dict(order=o['id'], disc=disc['id'], amt=amt))
    else:  # Line scope -> order_product_discount
        line = random.choice(lines)
        line_gross = line['qty']*line['gross']
        if disc['dtype'] == 'Percentage':
            amt = line_gross * disc['val']/100.0
        elif disc['dtype'] == 'Fixed Amount':
            amt = min(disc['val'], line_gross*0.5)
        else:
            amt = round(random.uniform(8,18),2)
        amt = round(min(amt, line_gross*0.9), 2)
        key = (o['id'], line['prod'], disc['id'])
        if amt >= 0 and key not in opd_seen:
            opd_seen.add(key)
            order_product_discount.append(dict(order=o['id'], prod=line['prod'],
                                               disc=disc['id'], amt=amt))

# ============================================================ VALIDATION
errs = []
for p in perf:
    if not (p['clicks'] < p['impr']): errs.append(('clicks<impr', p['id']))
    if not (p['conv'] < p['clicks']): errs.append(('conv<clicks', p['id']))
for op in order_products:
    if not (op['gross'] > 0): errs.append(('gross>0', op['order']))
print("constraint violations:", len(errs), errs[:5])

print("\n--- ROW COUNTS ---")
for nm, arr in [('customers',customers),('products',products),('campaigns',campaigns),
                ('advertisements',advertisements),('advertisement_performance',perf),
                ('interaction_events',interaction_events),('orders',orders),
                ('order_products',order_products),('advertisement_order',advertisement_order),
                ('discounts',discounts),('order_product_discount',order_product_discount),
                ('order_discount',order_discount)]:
    print(f"{nm:28s}{len(arr):>7d}")

# quick analytic preview (mirrors a few queries) -------------------------------
import collections
# Q1-ish CTR by platform
agg = collections.defaultdict(lambda:[0,0,0,0.0])
for p in perf:
    pl = ad_by_id[p['ad']]['plat']
    agg[pl][0]+=p['impr']; agg[pl][1]+=p['clicks']; agg[pl][2]+=p['conv']; agg[pl][3]+=p['rev']
print("\n--- platform funnel (CTR / click->conv / ROAS-ish vs spend) ---")
spend_by_plat = collections.defaultdict(float)
for a in advertisements: spend_by_plat[a['plat']] += a['budget']
for pl,(im,cl,cv,rv) in agg.items():
    print(f"{pl:10s} CTR {cl/im*100:5.2f}%  C2C {cv/cl*100:5.2f}%  "
          f"CPA ${spend_by_plat[pl]/max(1,cv):6.2f}  platRev/spend {rv/spend_by_plat[pl]:4.2f}")

# EOFY YoY revenue (actual sales attributed to EOFY campaigns)
print("\n--- EOFY campaign actual attributed sales by year ---")
op_lookup = {(op['order'],op['prod']):op for op in order_products}
order_rev = collections.defaultdict(float)
for op in order_products:
    if orders_by_id[op['order']]['status']=='Completed':
        order_rev[op['order']] += op['qty']*op['gross']
eofy_year_rev = collections.defaultdict(float)
ao_by_order = collections.defaultdict(list)
for ao in advertisement_order: ao_by_order[ao['order']].append(ao['ad'])
for o in orders:
    if o['status']!='Completed': continue
    cset=set()
    for ad in ao_by_order.get(o['id'],[]):
        c=ad_by_id[ad]['camp']
        if camp_by_id[c]['is_eofy']: cset.add(camp_by_id[c]['year'])
    for y in cset:
        eofy_year_rev[y]+=order_rev[o['id']]
for y in sorted(eofy_year_rev): print(f"  EOFY {y}: ${eofy_year_rev[y]:,.0f}")

# ============================================================ EMIT SQL
def s(v):
    if v is None: return 'NULL'
    if isinstance(v,(int,)): return str(v)
    if isinstance(v,float): return f"{v:.2f}"
    return "'" + str(v).replace("\\","\\\\").replace("'","''") + "'"

def emit(f, table, cols, rows, rowfmt, batch=200):
    f.write(f"\n-- {table}: {len(rows)} rows\n")
    for i in range(0,len(rows),batch):
        chunk = rows[i:i+batch]
        f.write(f"INSERT INTO {table} ({', '.join(cols)}) VALUES\n")
        f.write(",\n".join(rowfmt(r) for r in chunk) + ";\n")

with open(OUTPUT_PATH,'w') as f:
    f.write("-- ================================================================\n")
    f.write("-- David Jones marketing database — synthetic dataset\n")
    f.write("-- Span 2021-01-01 .. 2026-05-31. Run AFTER Entities.sql.\n")
    f.write("-- Reproducible (seed=42). Generated for EOFY 2026 planning analysis.\n")
    f.write("-- ================================================================\n")
    f.write("SET FOREIGN_KEY_CHECKS=0;\n")

    emit(f,'customers',
         ['customer_id','customer_name','customer_email','location','phone_number','referrer_id','signup_date','loyalty_tier'],
         customers, lambda c:f"({c['id']}, {s(c['name'])}, {s(c['email'])}, {s(c['loc'])}, {s(c['phone'])}, {s(c['ref'])}, {s(d(c['signup']))}, {s(c['tier'])})")

    emit(f,'products',
         ['product_id','product_name','category','brand','unit_price','unit_cost','stock_quantity'],
         products, lambda p:f"({p['id']}, {s(p['name'])}, {s(p['cat'])}, {s(p['brand'])}, {s(p['price'])}, {s(p['cost'])}, {p['stock']})")

    emit(f,'campaigns',
         ['campaign_id','campaign_name','objective','total_budget','start_date','end_date'],
         campaigns, lambda c:f"({c['id']}, {s(c['name'])}, {s(c['obj'])}, {s(c['budget'])}, {s(d(c['start']))}, {s(d(c['end']))})")

    emit(f,'advertisements',
         ['ad_id','campaign_id','platform','allocated_budget','start_date','end_date'],
         advertisements, lambda a:f"({a['id']}, {a['camp']}, {s(a['plat'])}, {s(a['budget'])}, {s(d(a['start']))}, {s(d(a['end']))})")

    emit(f,'advertisement_performance',
         ['ad_performance_id','ad_id','reported_date','device','impressions','clicks','conversions','platform_ad_revenue'],
         perf, lambda p:f"({p['id']}, {p['ad']}, {s(d(p['rep']))}, {s(p['dev'])}, {p['impr']}, {p['clicks']}, {p['conv']}, {s(p['rev'])})")

    emit(f,'interaction_events',
         ['interaction_id','ad_id','customer_id','duration_seconds','interaction_datetime','device'],
         interaction_events, lambda x:f"({x['id']}, {x['ad']}, {x['cust']}, {s(x['dur'])}, {s(dt(x['dt']))}, {s(x['dev'])})")

    emit(f,'orders',
         ['order_id','customer_id','order_date','payment_method','order_status'],
         orders, lambda o:f"({o['id']}, {o['cust']}, {s(dt(o['dt']))}, {s(o['pay'])}, {s(o['status'])})")

    emit(f,'order_products',
         ['order_id','product_id','quantity','gross_price'],
         order_products, lambda op:f"({op['order']}, {op['prod']}, {op['qty']}, {s(op['gross'])})")

    emit(f,'advertisement_order',
         ['ad_id','order_id','clicked_date'],
         advertisement_order, lambda a:f"({a['ad']}, {a['order']}, {s(d(a['clicked']))})")

    emit(f,'discounts',
         ['discount_id','campaign_id','discount_name','discount_type','scope','discount_value','promo_code','min_spend','discount_start_date','discount_end_date'],
         discounts, lambda x:f"({x['id']}, {s(x['camp'])}, {s(x['name'])}, {s(x['dtype'])}, {s(x['scope'])}, {s(x['val'])}, {s(x['promo'])}, {s(x['min_spend'])}, {s(d(x['start']))}, {s(d(x['end']))})")

    emit(f,'order_product_discount',
         ['order_id','product_id','discount_id','product_discount_amount'],
         order_product_discount, lambda x:f"({x['order']}, {x['prod']}, {x['disc']}, {s(x['amt'])})")

    emit(f,'order_discount',
         ['order_id','discount_id','order_discount_amount'],
         order_discount, lambda x:f"({x['order']}, {x['disc']}, {s(x['amt'])})")

    f.write("\nSET FOREIGN_KEY_CHECKS=1;\n")

import os
print("\n"+OUTPUT_PATH+" size:", round(os.path.getsize(OUTPUT_PATH)/1024/1024,2), "MB")

constraint violations: 0 []

--- ROW COUNTS ---
customers                       400
products                        100
campaigns                        32
advertisements                  170
advertisement_performance      1906
interaction_events            15000
orders                        10000
order_products                18347
advertisement_order            2827
discounts                        50
order_product_discount          894
order_discount                  910

--- platform funnel (CTR / click->conv / ROAS-ish vs spend) ---
WeChat     CTR  3.07%  C2C 24.98%  CPA $  2.72  platRev/spend 5.10
Instagram  CTR  6.02%  C2C 15.02%  CPA $  1.99  platRev/spend 3.03
Facebook   CTR  3.98%  C2C 18.45%  CPA $  2.59  platRev/spend 2.70
X          CTR  2.10%  C2C  9.16%  CPA $ 11.91  platRev/spend 0.34

--- EOFY campaign actual attributed sales by year ---
  EOFY 2021: $13,741
  EOFY 2022: $57,268
  EOFY 2023: $57,361
  EOFY 2024: $129,337
  EOFY 2025: $145,021

Data.sql size: 2.24 MB


In [5]:
import csv, os
CSV_DIR = "csv"
os.makedirs(CSV_DIR, exist_ok=True)
 
def write_csv(name, header, rows, rowvals):
    with open(os.path.join(CSV_DIR, name + ".csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(header)
        for r in rows:
            w.writerow(["" if v is None else v for v in rowvals(r)])
 
write_csv("customers",
    ["customer_id","customer_name","customer_email","location","phone_number","referrer_id","signup_date","loyalty_tier"],
    customers, lambda c:[c['id'],c['name'],c['email'],c['loc'],c['phone'],c['ref'],d(c['signup']),c['tier']])
write_csv("products",
    ["product_id","product_name","category","brand","unit_price","unit_cost","stock_quantity"],
    products, lambda p:[p['id'],p['name'],p['cat'],p['brand'],p['price'],p['cost'],p['stock']])
write_csv("campaigns",
    ["campaign_id","campaign_name","objective","total_budget","start_date","end_date"],
    campaigns, lambda c:[c['id'],c['name'],c['obj'],c['budget'],d(c['start']),d(c['end'])])
write_csv("advertisements",
    ["ad_id","campaign_id","platform","allocated_budget","start_date","end_date"],
    advertisements, lambda a:[a['id'],a['camp'],a['plat'],a['budget'],d(a['start']),d(a['end'])])
write_csv("advertisement_performance",
    ["ad_performance_id","ad_id","reported_date","device","impressions","clicks","conversions","platform_ad_revenue"],
    perf, lambda p:[p['id'],p['ad'],d(p['rep']),p['dev'],p['impr'],p['clicks'],p['conv'],p['rev']])
write_csv("interaction_events",
    ["interaction_id","ad_id","customer_id","duration_seconds","interaction_datetime","device"],
    interaction_events, lambda x:[x['id'],x['ad'],x['cust'],x['dur'],dt(x['dt']),x['dev']])
write_csv("orders",
    ["order_id","customer_id","order_date","payment_method","order_status"],
    orders, lambda o:[o['id'],o['cust'],dt(o['dt']),o['pay'],o['status']])
write_csv("order_products",
    ["order_id","product_id","quantity","gross_price"],
    order_products, lambda op:[op['order'],op['prod'],op['qty'],op['gross']])
write_csv("advertisement_order",
    ["ad_id","order_id","clicked_date"],
    advertisement_order, lambda a:[a['ad'],a['order'],d(a['clicked'])])
write_csv("discounts",
    ["discount_id","campaign_id","discount_name","discount_type","scope","discount_value","promo_code","min_spend","discount_start_date","discount_end_date"],
    discounts, lambda x:[x['id'],x['camp'],x['name'],x['dtype'],x['scope'],x['val'],x['promo'],x['min_spend'],d(x['start']),d(x['end'])])
write_csv("order_product_discount",
    ["order_id","product_id","discount_id","product_discount_amount"],
    order_product_discount, lambda x:[x['order'],x['prod'],x['disc'],x['amt']])
write_csv("order_discount",
    ["order_id","discount_id","order_discount_amount"],
    order_discount, lambda x:[x['order'],x['disc'],x['amt']])
print("CSV files written to ./" + CSV_DIR + "/")

CSV files written to ./csv/
